<a href="https://colab.research.google.com/github/irsyad-kamil/indonesia-ai-nlpa-summary/blob/bert2bert-model/Splitting_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1) Environment Setup
Mount Google Drive and install required libraries.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q transformers datasets evaluate rouge_score

## 2) Data Extraction and Sampling
Extract raw data, refresh destination folders, and build train/dev/test samples.

In [ ]:
import os

file_path = '/content/drive/MyDrive/Bootcamp/Project 2/liputan6_data.tar.gz'
local_extract_path = '/content/temp_extract'

# Create local directory
os.makedirs(local_extract_path, exist_ok=True)

print(f"Extracting 'canonical' folder to local storage {local_extract_path}...")
# Extract only the canonical directory from the archive
!tar -xzf "{file_path}" -C "{local_extract_path}" liputan6_data/canonical
print("Local extraction finished.")

Extracting 'canonical' folder to local storage /content/temp_extract...
Local extraction finished.


In [ ]:
import os
import json
import pandas as pd
import random
from tqdm.notebook import tqdm

def load_and_split_sampled(base_path, total_sample=10000, train_ratio=0.7, dev_ratio=0.1):
    # 1. Kumpulkan SEMUA path file dari folder train asli
    train_dir = os.path.join(base_path, 'train')
    all_files = [os.path.join(train_dir, f) for f in os.listdir(train_dir) if f.endswith('.json')]

    # 2. Acak dan Ambil sesuai limit (10.000)
    random.seed(42)
    random.shuffle(all_files)
    sampled_paths = all_files[:total_sample]

    # 3. Hitung batasan split
    train_end = int(total_sample * train_ratio)
    dev_end = train_end + int(total_sample * dev_ratio)

    splits = {
        'train': sampled_paths[:train_end],
        'dev': sampled_paths[train_end:dev_end],
        'test': sampled_paths[dev_end:]
    }

    results = {}

    for name, paths in splits.items():
        records = []
        for p in tqdm(paths, desc=f"Processing {name}"):
            with open(p, 'r') as f:
                data = json.load(f)
                # Pakai logika join kalimat temanmu yang bagus tadi
                article = " ".join([" ".join(s) for s in data.get('clean_article', [])])
                summary = " ".join([" ".join(s) for s in data.get('clean_summary', [])])

                records.append({
                    'id': data.get('id'),
                    'article': article,
                    'summary': summary
                })
        results[name] = pd.DataFrame(records)

    return results['train'], results['dev'], results['test']

# Eksekusi
base_dir = '/content/drive/MyDrive/Bootcamp/Project 2/liputan6_data/canonical' # Sesuaikan path ekstrakmu
df_train, df_dev, df_test = load_and_split_sampled(base_dir, total_sample=10000)

Processing train:   0%|          | 0/7000 [00:00<?, ?it/s]

Processing dev:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing test:   0%|          | 0/2000 [00:00<?, ?it/s]

In [ ]:
# Fungsi untuk menggabungkan list menjadi string tunggal
# Kita tambahkan pengecekan isinstance untuk memastikan hanya memproses data yang berbentuk list
df_train['article'] = df_train['article'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)
df_train['summary'] = df_train['summary'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)

# Lakukan hal yang sama untuk df_dev dan df_test jika ada
df_dev['article'] = df_dev['article'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)
df_dev['summary'] = df_dev['summary'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)

df_test['article'] = df_test['article'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)
df_test['summary'] = df_test['summary'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)

# Cek hasil 1 data teratas
print("Contoh artikel setelah digabung:")
print(df_train['article'].iloc[0][:200] + "...") # Tampilkan 200 karakter pertama saja

Contoh artikel setelah digabung:
Liputan6 . com , Jakarta : Permintaan Anggodo Widjojo untuk menginap di Gedung Dewan Pertimbangan Presiden ( Wantimpres ) , tempat Tim Pencari Fakta kasus Bibit-Chandra ( TPF ) bekerja , tampaknya ter...


In [ ]:
# Tentukan folder penyimpanan di Drive
save_path = '/content/drive/MyDrive/Bootcamp/Project 2/processed_data'

# Buat folder jika belum ada
if not os.path.exists(save_path):
    os.makedirs(save_path)

# Simpan ke CSV
df_train.to_csv(f'{save_path}/liputan6_10k_train.csv', index=False)
df_dev.to_csv(f'{save_path}/liputan6_10k_dev.csv', index=False)
df_test.to_csv(f'{save_path}/liputan6_10k_test.csv', index=False)

print(f"Berhasil menyimpan 3 file CSV di: {save_path}")

Berhasil menyimpan 3 file CSV di: /content/drive/MyDrive/Bootcamp/Project 2/processed_data


In [ ]:
import os
import json
import pandas as pd
import random
from tqdm.notebook import tqdm

df_train = pd.read_csv('/content/drive/MyDrive/Bootcamp/Project 2/processed_data/liputan6_10k_train.csv')
df_dev = pd.read_csv('/content/drive/MyDrive/Bootcamp/Project 2/processed_data/liputan6_10k_dev.csv')
df_test = pd.read_csv('/content/drive/MyDrive/Bootcamp/Project 2/processed_data/liputan6_10k_test.csv')